# 04: Organize Block Model Columns into Groups

This tutorial shows how to organize block model columns into groups with the high-level Python SDK: create groups, add columns directly into them, move columns between groups, and ungroup them.

## What You Will Learn

- Create top-level and nested groups
- Add new columns directly into a group
- Create a group and move existing columns into it in a single version
- Move a column and replace its data in the same update
- Ungroup a column

## Key Concepts

- A column's **title** is its own name, for example `Cu`.
- A column's **qualified title** joins the group and the title with `▸`, for example `Assays▸Cu`. Nested groups chain the same way: `Assays▸Geochem▸Au`.
- When you upload data for a grouped column, key the data by its **qualified title**. The SDK uploads the data as given and never renames columns for you; `qualify_column_titles` is an optional helper that does it.
- To change a column that already exists, address it by its **current** title. If you also upload data for a column that moves, key that data by its **destination** qualified title (the bare title when ungrouping).
- Every call below creates exactly one new block model version, even when it combines several changes.

## Before You Start

You need an Evo app client ID and redirect URL, and permission to create block models in the selected workspace. This notebook creates its own block model and does not change existing ones.

Column groups are currently part of the Block Model Service preview API, so the client is created with `preview=True`.

## Learning Path

1. [01: Create and Query a Regular Block Model](01-create-and-query-regular-block-model.ipynb)
2. [02: Transform an Existing Block Model Attribute](02-transform-existing-block-model-attribute.ipynb)
3. [03: Block Model Reports](03-block-model-reports.ipynb)
4. **This notebook**: organize columns into groups

In [ ]:
from evo.notebooks import ServiceManagerWidget

# Evo app credentials
client_id = "your-client-id"  # Replace with your client ID
redirect_url = "your-redirect-url"  # Replace with your redirect URL

manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id,
    redirect_url=redirect_url,
).login()

In [ ]:
from evo.blockmodels import BlockModelAPIClient

environment = manager.get_environment()
connector = manager.get_connector()

# Column groups are a preview feature, so opt in to the preview API.
service_client = BlockModelAPIClient(environment, connector, manager.cache, preview=True)

### Demo 1: Create a small block model

The block model is a 2 x 2 x 2 grid (8 blocks) with three ungrouped columns.

In [ ]:
from datetime import datetime

import pyarrow

from evo.blockmodels.data import RegularGridDefinition
from evo.blockmodels.endpoints.models import RotationAxis

grid_definition = RegularGridDefinition(
    model_origin=[0, 0, 0],
    rotations=[(RotationAxis.x, 0), (RotationAxis.y, 0), (RotationAxis.z, 0)],
    n_blocks=[2, 2, 2],
    block_size=[10, 10, 10],
)

# Block centroids for the 2 x 2 x 2 grid.
centres = [(x, y, z) for x in (5.0, 15.0) for y in (5.0, 15.0) for z in (5.0, 15.0)]
n_blocks = len(centres)
geometry = {
    "x": [c[0] for c in centres],
    "y": [c[1] for c in centres],
    "z": [c[2] for c in centres],
}


def make_table(**columns):
    """Build a table of block centroids plus the given columns (constant value per column)."""
    return pyarrow.table({**geometry, **{title: [value] * n_blocks for title, value in columns.items()}})


block_model, version = await service_client.create_block_model(
    name=f"Column groups {datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')}",
    description="Column groups tutorial",
    grid_definition=grid_definition,
    object_path="/Jupyter/SDK-demo",
    coordinate_reference_system="EPSG:3395",
    size_unit_id="m",
    initial_data=make_table(Cu=1.0, Au=2.0, Fe=3.0),
)

print(f"Created block model {block_model.id} (version {version.version_id})")

This helper prints each column with the qualified title of its group, so you can see the effect of every step.

In [ ]:
def show_groups(version):
    print(f"Version {version.version_id}")
    for column in version.columns:
        if column.title in {"x", "y", "z", "i", "j", "k"}:
            continue
        group = version.group_for_column(column)
        group_title = version.qualified_group_title(group) if group else "(ungrouped)"
        print(f"  {column.title:<8} -> {group_title}")


show_groups(version)

### Demo 2: Create groups

Groups are created with `update_groups`. A nested group names its parent with `parent_group`.

In [ ]:
from evo.blockmodels.data import GroupDefinition

version = await service_client.update_groups(
    block_model.id,
    new=[
        GroupDefinition(title="Assays"),
        GroupDefinition(title="Geochem", parent_group="Assays"),
    ],
    comment="Create Assays and Assays▸Geochem",
)

for group in version.groups:
    print(version.qualified_group_title(group))

### Demo 3: Add new columns directly into groups

`qualify_column_titles` renames the table columns to their qualified titles and returns the matching `column_groups` mapping. Here `Zn` goes into `Assays` and `Ag` into the nested `Assays▸Geochem` group.

In [ ]:
from evo.blockmodels import get_qualified_title, qualify_column_titles

geochem = get_qualified_title("Assays", "Geochem")  # "Assays▸Geochem"

data, column_groups = qualify_column_titles(
    make_table(Zn=4.0, Ag=5.0),
    {"Zn": "Assays", "Ag": geochem},
)

version = await service_client.add_new_columns(
    block_model.id,
    data=data,
    column_groups=column_groups,
)

show_groups(version)

### Demo 4: Create a group and move existing columns into it (metadata only)

`update_groups` can create a group and move existing columns in a single call, with no data upload. `move_columns` maps a column's current title to the destination group; use `""` to ungroup. Moves that don't upload data only change the column's membership; its data is untouched.

In [ ]:
version = await service_client.update_groups(
    block_model.id,
    new=[GroupDefinition(title="Density")],
    move_columns={"Fe": "Density"},
    comment="Create Density and move Fe into it",
)

show_groups(version)

### Demo 5: Create a group, move a column and replace its data in one update

When the data changes at the same time, use the column update methods. They accept:

- `new_groups`: groups to create in this update
- `column_metadata_updates`: per-column changes, such as the destination `group` (also `unit_id` and `tags`)

The column is listed in `update_columns` under its **current** title (`Cu`), but its data is keyed by its **destination** qualified title (`Metals▸Cu`).

In [ ]:
from evo.blockmodels.data import ColumnMetadataUpdate

version = await service_client.update_block_model_columns(
    block_model.id,
    data=make_table(**{get_qualified_title("Metals", "Cu"): 10.0}),
    new_columns=[],
    update_columns={"Cu"},  # the column's current title
    new_groups=[GroupDefinition(title="Metals")],
    column_metadata_updates={"Cu": ColumnMetadataUpdate(group="Metals")},
)

show_groups(version)

### Demo 6: Ungroup a column

An empty `group` ungroups the column. The column is addressed by its current qualified title, and any data you upload is keyed by the bare title.

In [ ]:
version = await service_client.update_groups(
    block_model.id,
    move_columns={get_qualified_title("Assays", "Zn"): ""},
    comment="Ungroup Zn",
)

show_groups(version)

### Demo 7: Query the grouped columns

Querying with `ColumnHeaderType.name` returns the column names.

In [ ]:
from evo.blockmodels.endpoints.models import ColumnHeaderType

table = await service_client.query_block_model_as_table(
    bm_id=block_model.id,
    columns=["*"],
    column_headers=ColumnHeaderType.name,
)

display(table.to_pandas())

## Notes

- The default separator is `▸`. Both write and query methods accept a `separator` argument if your column titles contain `▸`; use the same separator when reading the model back.
- `update_block_model_columns` and `update_subblocked_columns` take the same `new_groups` and `column_metadata_updates` arguments.
- A column cannot be both moved and deleted in the same update.